In [ ]:
from dotenv import load_dotenv

MODEL = "claude-haiku-4-5"
DB_NAME = "vector_db"
load_dotenv(override=True)

In [ ]:
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="google/embeddinggemma-300m")
vectorstore = Chroma(persist_directory=DB_NAME, embedding_function=embeddings)

In [ ]:
# from langchain_openai import ChatOpenAI
from langchain_anthropic import ChatAnthropic

retriever = vectorstore.as_retriever()
llm = ChatAnthropic(temperature=0, model_name=MODEL)

In [ ]:
retriever.invoke("Who is Avery?")

In [ ]:
llm.invoke("Who is Avery?")

In [ ]:
SYSTEM_PROMPT_TEMPLATE = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [ ]:
from langchain_core.messages import SystemMessage, HumanMessage

def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT_TEMPLATE.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [ ]:
answer_question("Who is Averi Lancaster?", [])

In [ ]:
import gradio as gr

gr.ChatInterface(answer_question).launch()